# Hamilton STAR

The Hamilton STAR(let) is a liquid handler with independent pipetting channels, an optional 96-head, and an optional iSWAP plate transport arm.

In this notebook, you will learn how to use PyLabRobot to set up the STAR, pick up tips, and move liquid between wells.

**Prerequisites:**

- Installed PyLabRobot with USB support: `pip install pylabrobot[usb]`
- Platform-specific driver setup (libusb on Mac, Zadig on Windows) — see [the installation guide](../../_getting-started/installation)
- Connected the Hamilton to your computer using the USB cable

Video of what this code does:

<iframe width="640" height="360" src="https://www.youtube.com/embed/NN6ltrRj3bU" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>

## Setup

Import {class}`~pylabrobot.hamilton.liquid_handlers.star.star.STAR` and one of the Hamilton deck classes. `STAR` is the device that owns the driver and exposes capabilities like `pip` (independent channels), `head96` (96-head), and `iswap` (plate transport arm).

In [1]:
from pylabrobot.hamilton.liquid_handlers.star import STAR
from pylabrobot.resources.hamilton import STARLetDeck

deck = STARLetDeck()
star = STAR(deck=deck)
await star.setup()

2026-04-02 14:25:26,755 - pylabrobot.io.usb - INFO - Finding USB device...
2026-04-02 14:25:26,759 - pylabrobot.io.usb - INFO - Found USB device.
2026-04-02 14:25:26,759 - pylabrobot.io.usb - INFO - Found endpoints. 
Write:
       ENDPOINT 0x2: Bulk OUT ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :    0x2 OUT
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0 
Read:
       ENDPOINT 0x81: Bulk IN ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :   0x81 IN
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0


After `setup()`, the driver discovers installed hardware automatically. `star.pip` is always available. `star.head96` and `star.iswap` are `None` if the corresponding hardware is not installed.

## Creating the deck layout

Define the physical deck layout by assigning carriers with tip racks and plates. This tutorial uses:

- {class}`~pylabrobot.resources.hamilton.tip_carriers.TIP_CAR_480_A00` tip carrier
- {class}`~pylabrobot.resources.hamilton.plate_carriers.PLT_CAR_L5AC_A00` plate carrier
- {class}`~pylabrobot.resources.corning_costar.plates.Cor_96_wellplate_360ul_Fb` 96-well plate
- {class}`~pylabrobot.resources.hamilton.tip_racks.hamilton_96_tiprack_1000uL_filter` tip rack

In [2]:
from pylabrobot.resources import (
  TIP_CAR_480_A00,
  PLT_CAR_L5AC_A00,
  Cor_96_wellplate_360ul_Fb,
  hamilton_96_tiprack_1000uL_filter,
)

tip_car = TIP_CAR_480_A00(name="tip carrier")
tip_car[0] = hamilton_96_tiprack_1000uL_filter(name="tips_01")
deck.assign_child_resource(tip_car, rails=3)

plt_car = PLT_CAR_L5AC_A00(name="plate carrier")
plt_car[0] = Cor_96_wellplate_360ul_Fb(name="plate_01")
deck.assign_child_resource(plt_car, rails=15)

Let's look at a summary of the deck layout using {meth}`~pylabrobot.resources.deck.Deck.summary`.

In [3]:
print(deck.summary())

Rail  Resource                      Type                 Coordinates (mm)
(-6)  ├── trash_core96              Trash                (-58.200, 106.000, 216.400)
      │
(3)   ├── tip carrier               TipCarrier           (145.000, 063.000, 100.000)
      │   ├── tips_01               TipRack              (151.200, 073.000, 214.950)
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │
(15)  ├── plate carrier             PlateCarrier         (415.000, 063.000, 100.000)
      │   ├── plate_01              Plate                (419.000, 071.500, 183.120)
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │
(31)  ├── waste_block               Resource             (775.000, 115.000, 100.000)
      │   ├── teaching_tip_rack     TipRack              (780.900, 461.100, 100.000)
      │   ├── core_grippers         HamiltonCoreGrippers (797.500, 085.500, 205.000)
      │
(32)  ├── trash                     Tr

## Picking up tips

Use `star.pip` — the independent-channel pipetting capability — to pick up tips. Indexing a tip rack with `["A1:C1"]` returns the first three tip spots in column 1.

In [ ]:
tiprack = deck.get_resource("tips_01")
await star.pip.pick_up_tips(tiprack["A1:C1"])

## Aspirating and dispensing

Aspirate from wells `A1:C1` and dispense to `D1:F1`, using channels 0, 1, and 2.

In [7]:
plate = deck.get_resource("plate_01")
await star.pip.aspirate(plate["A1:C1"], vols=[100.0, 50.0, 200.0])

In [8]:
await star.pip.dispense(plate["D1:F1"], vols=[100.0, 50.0, 200.0])

Move the liquid back:

In [9]:
await star.pip.aspirate(plate["D1:F1"], vols=[100.0, 50.0, 200.0])
await star.pip.dispense(plate["A1:C1"], vols=[100.0, 50.0, 200.0])

## Dropping tips

Return tips to their original positions:

In [ ]:
await star.pip.drop_tips(tiprack["A1:C1"])

In [ ]:
await star.stop()
await star.setup()

In [9]:
await star.driver.send_command("C0", "RT")

'C0RTid0031er00/00rt0 0 0 0 0 0 0 0 0 0 0 0'

In [ ]:
import asyncio

channels = star.driver.pip.channels
results = await asyncio.gather(*[
  channels[i].request_probe_z_position() for i in range(3)
])
results

## Backend parameters

For STAR-specific tuning, pass `backend_params` to any operation. The available parameter classes are:

- {class}`~pylabrobot.hamilton.liquid_handlers.star.pip_backend.STARPIPBackend.PickUpTipsParams`
- {class}`~pylabrobot.hamilton.liquid_handlers.star.pip_backend.STARPIPBackend.DropTipsParams`
- {class}`~pylabrobot.hamilton.liquid_handlers.star.pip_backend.STARPIPBackend.AspirateParams`
- {class}`~pylabrobot.hamilton.liquid_handlers.star.pip_backend.STARPIPBackend.DispenseParams`

For example, to use liquid level detection during aspiration:

In [11]:
from pylabrobot.hamilton.liquid_handlers.star.pip_backend import STARPIPBackend, LLDMode

await star.pip.pick_up_tips(tiprack["D1:F1"])

await star.pip.aspirate(
  plate["A1:C1"],
  vols=[100.0, 100.0, 100.0],
  backend_params=STARPIPBackend.AspirateParams(
    lld_mode=[LLDMode.GAMMA, LLDMode.GAMMA, LLDMode.GAMMA],
  ),
)
await star.pip.dispense(plate["D1:F1"], vols=[100.0, 100.0, 100.0])

await star.pip.drop_tips(tiprack["D1:F1"])

ChannelizedError: ChannelizedError(errors={0: TooLittleLiquidError('No liquid level found (possibly because no liquid was present, or too little liquid was present to trigger cLLD)'), 1: TooLittleLiquidError('No liquid level found (possibly because no liquid was present, or too little liquid was present to trigger cLLD)'), 2: TooLittleLiquidError('No liquid level found (possibly because no liquid was present, or too little liquid was present to trigger cLLD)')}, raw_response=C0ASid0029er99/00 P106/70 P206/70 P306/70)

In [ ]:
await star.pip.drop_tips(tiprack["D1:F1"])

## Teardown

In [ ]:
await star.stop()